# RetailPulse — Database Quality Checks
This notebook validates the deployable SQLite database and its analytical views.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB = ROOT / 'data' / 'customer_sales.db'
conn = sqlite3.connect(DB)


In [ ]:
objects = pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type IN ('table','view') AND name NOT LIKE 'sqlite_%' ORDER BY type, name", conn)
objects


In [ ]:
tables = ['salespersons','customers','products','orders','order_items']
counts = {t: pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {t}', conn).loc[0,'n'] for t in tables}
pd.Series(counts, name='row_count')


In [ ]:
pd.read_sql_query('PRAGMA foreign_key_check', conn)


In [ ]:
facts = pd.read_sql_query('SELECT * FROM vw_sales_facts', conn)
facts[['quantity','unit_price','line_revenue']].describe()


In [ ]:
assert len(pd.read_sql_query('PRAGMA foreign_key_check', conn)) == 0
assert facts['line_revenue'].sum() > 0
print('All checks passed.')
